# Scraping farmacias

Prepara y ejecuta el pipeline semanal de precios cuando `RUN` sea `True`.

In [ ]:
from pathlib import Path
import subprocess
import pandas as pd

ROOT = Path.cwd()
if (ROOT / 'Suplematch-Backend').exists():
    ROOT = ROOT / 'Suplematch-Backend'
while ROOT.name != 'Suplematch-Backend' and ROOT.parent != ROOT:
    ROOT = ROOT.parent

SCRIPT = ROOT / 'scripts/scraping/run_weekly_supplement_update.sh'
assert SCRIPT.exists()
ROOT

## Entradas y salidas

In [ ]:
paths = {
    'raw': ROOT / 'data/raw/pharmacies/supplements_exhaustive_clean.csv',
    'approved': ROOT / 'data/catalog/approved_catalog.csv',
    'rejected': ROOT / 'data/reports/scraping/supplements_rejected.csv',
    'report': ROOT / 'data/reports/scraping/catalog_pipeline_current_report.json',
}
pd.DataFrame([{'name': key, 'path': str(path.relative_to(ROOT)), 'exists': path.exists(), 'size': path.stat().st_size if path.exists() else 0} for key, path in paths.items()])

## Comando

In [ ]:
RUN = False
command = ['bash', str(SCRIPT)]
result = None
if RUN:
    result = subprocess.run(command, cwd=ROOT, text=True, capture_output=True, check=True)
{'command': ' '.join(command), 'executed': RUN, 'stdout': result.stdout[-2000:] if result else ''}

## Lectura de resultados

In [ ]:
approved = paths['approved']
if approved.exists():
    df = pd.read_csv(approved, keep_default_na=False)
    display(df.head(10))
    display(df.groupby('pharmacy').size().reset_index(name='products').sort_values('products', ascending=False))
else:
    pd.DataFrame()